Colab 4 — Reinforcement Learning with GRPO (Reasoning Model)

In [1]:
import torch, os
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
!nvidia-smi -L || echo "No GPU detected — enable one in Runtime → Change runtime type → GPU."


Torch: 2.8.0+cu126
CUDA available: True
GPU 0: NVIDIA A100-SXM4-40GB (UUID: GPU-1d6b3fcd-e056-856a-102b-d25c49d88f6e)


In [2]:
!pip install -qU "unsloth>=2025.9.0" "transformers>=4.45.0" "trl>=0.10.0" \
 "datasets>=2.20.0" "accelerate>=1.0.0" "bitsandbytes>=0.43.0" "peft>=0.13.0"


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.8/61.8 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 351.3/351.3 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 564.7/564.7 kB 27.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 44.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.4/59.4 MB 44.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 MB 55.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 24.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.2/117.2 MB 22.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.6/132.6 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 143.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 213.6/213.6 kB 16.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behav

In [3]:
from unsloth import FastLanguageModel
import torch

BASE_MODEL = "HuggingFaceTB/SmolLM2-135M-Instruct"
MAX_SEQ_LEN = 1024
dtype = torch.bfloat16 if (torch.cuda.is_available() and torch.cuda.is_bf16_supported()) else torch.float16

policy, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LEN,
    dtype=dtype,
    load_in_4bit=False,
)
policy = FastLanguageModel.get_peft_model(
    policy,
    r=16, lora_alpha=16, lora_dropout=0.0,
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    use_gradient_checkpointing=True,
)
tokenizer.pad_token = tokenizer.eos_token


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2025.11.2: Fast Llama patching. Transformers: 4.57.1.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.557 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu126. CUDA: 8.0. CUDA Toolkit: 12.6. Triton: 3.4.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.32.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/269M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

HuggingFaceTB/SmolLM2-135M-Instruct does not have a padding token! Will use pad_token = <|endoftext|>.


Unsloth 2025.11.2 patched 30 layers with 30 QKV layers, 30 O layers and 30 MLP layers.


In [4]:
from datasets import Dataset

samples = [
    {"prompt": "If 3 apples cost $6, how much do 5 apples cost?",
     "answer": "Reasoning: Each apple costs $2. So 5 apples cost 5×2 = $10. Final Answer: $10"},
    {"prompt": "John has twice as many pens as Mary. Together they have 18 pens. How many pens does each have?",
     "answer": "Reasoning: Let Mary's pens = x ⇒ John's = 2x ⇒ 3x = 18 ⇒ x = 6 ⇒ Mary = 6, John = 12. Final Answer: John 12, Mary 6"},
]
reason_ds = Dataset.from_list(samples)


In [6]:
# ===== GRPO-style reasoning training (pure PyTorch, no GRPOTrainer) =====
import torch, copy, re, random
from datasets import Dataset
torch.backends.cuda.matmul.allow_tf32 = True

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
assert torch.cuda.is_available(), "❌ Need a GPU runtime."

# --- Ensure tokenizer/model padding ---
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id
try:
    policy.config.pad_token_id = tokenizer.pad_token_id
except Exception:
    pass
policy.to(device).train()
tokenizer.padding_side = "right"

MAX_PROMPT_LEN     = 256
MAX_COMPLETION_LEN = 256
MAX_SEQ_LEN        = 1024

# --- If you don't already have a reasoning dataset, make a tiny one ---
if "reason_ds" not in globals():
    samples = [
        {"prompt": "If 3 apples cost $6, how much do 5 apples cost?",
         "answer": "$10"},
        {"prompt": "John has twice as many pens as Mary. Together they have 18 pens. How many pens does John have?",
         "answer": "12"},
        {"prompt": "What is 17 * 13?", "answer": "221"},
        {"prompt": "A train travels 60 km in 1.5 hours. What is its average speed (km/h)?", "answer": "40"},
    ]
    reason_ds = Dataset.from_list(samples)

# --- Helper: build chat prompt that invites reasoning ---
def build_prompt_text(prompt: str) -> str:
    msgs = [{"role":"user","content": f"{prompt}\nPlease show your reasoning then give a line starting with 'Final Answer:'"}]
    return tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

# --- Reward function: +1 for correct final answer (string contains), small length penalty ---
final_ans_re = re.compile(r"final answer\s*:\s*(.*)", re.IGNORECASE)

def extract_final_answer(text: str) -> str:
    m = final_ans_re.search(text)
    if m:
        return m.group(1).strip().rstrip(".")
    return ""

def reward_fn(gen_text: str, gold_ans: str) -> float:
    pred = extract_final_answer(gen_text)
    correct = 1.0 if (pred and gold_ans.lower() in pred.lower()) else 0.0
    # light penalty for very long outputs
    length_penalty = 0.002 * max(0, len(gen_text) - 200)
    return max(0.0, correct - length_penalty)

# --- Frozen reference model for KL regularization ---
ref_model = copy.deepcopy(policy).to(device).eval()
for p in ref_model.parameters(): p.requires_grad = False

# --- Trainable params: only LoRA adapters require grad ---
trainable = [p for p in policy.parameters() if p.requires_grad]
optimizer = torch.optim.AdamW(trainable, lr=5e-5)
kl_coef   = 0.02      # weight for KL(policy || ref)
beta_grpo = 1.0       # scale for group-relative advantage (you can tune)

# --- Sampling config ---
K = 3                 # candidates per prompt
temperature = 0.7
top_p = 0.9

# --- Utilities to compute logprob of generated tokens ---
def concat_and_score_logprobs(model, prompt_ids, gen_ids):
    """Return summed log p(y|x) over generated tokens y."""
    # concat prompt + generated
    input_ids = torch.cat([prompt_ids, gen_ids], dim=1)
    attn = torch.ones_like(input_ids, device=input_ids.device)
    out = model(input_ids=input_ids, attention_mask=attn, use_cache=False)
    logits = out.logits[:, :-1, :]                      # shift
    targets = input_ids[:, 1:]
    logp = torch.log_softmax(logits, dim=-1)
    token_logp = logp.gather(-1, targets.unsqueeze(-1)).squeeze(-1)
    # only sum the generated region (exclude prompt positions)
    prompt_len = prompt_ids.shape[1]
    gen_region = token_logp[:, prompt_len-1:]           # -1 due to shift
    return gen_region.sum(dim=1)                        # [B]

# --- Mini-batch over prompts (1 prompt per step; K samples per prompt) ---
max_steps = 300
log_every = 10
random_indices = list(range(len(reason_ds)))
random.shuffle(random_indices)

step = 0
policy.train()

while step < max_steps:
    for idx in random_indices:
        if step >= max_steps: break
        ex = reason_ds[idx]
        gold = ex["answer"]
        # 1) Build prompt ids
        prompt_text = build_prompt_text(ex["prompt"])
        prompt_tok = tokenizer([prompt_text], truncation=True, max_length=MAX_PROMPT_LEN, return_tensors="pt").to(device)
        prompt_ids = prompt_tok.input_ids

        # 2) Sample K completions from policy
        with torch.no_grad():
            gen_out = policy.generate(
                input_ids=prompt_ids.repeat(K,1),
                max_new_tokens=MAX_COMPLETION_LEN,
                do_sample=True,
                temperature=temperature,
                top_p=top_p,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
            )
        # Split off generated-only piece
        gen_ids = gen_out[:, prompt_ids.shape[1]:]

        # 3) Decode and compute rewards per sample
        texts = tokenizer.batch_decode(gen_out, skip_special_tokens=True)
        rewards = torch.tensor([reward_fn(t, gold) for t in texts], device=device, dtype=torch.float32)

        # 4) Compute log probs under policy (with grad) and reference (no grad)
        logp_policy = concat_and_score_logprobs(policy, prompt_ids.repeat(K,1), gen_ids)    # [K]
        with torch.no_grad():
            logp_ref = concat_and_score_logprobs(ref_model, prompt_ids.repeat(K,1), gen_ids) # [K]

        # 5) GRPO objective:
        # group-center rewards (per prompt), then weight log p by (r - mean(r))
        centered = rewards - rewards.mean()
        # KL term encourages policy to stay near reference on generated tokens
        kl = (logp_policy - logp_ref).mean()
        # Loss = - E[(r - mean(r)) * logp] + lambda * KL
        loss = -(beta_grpo * (centered.detach() * logp_policy).mean()) + kl_coef * kl

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(trainable, 1.0)
        optimizer.step()

        if step % log_every == 0:
            acc = (rewards > 0.5).float().mean().item()
            print(f"step {step:4d} | loss {loss.item():.4f} | reward_mean {rewards.mean().item():.3f} | acc {acc:.2f} | kl {kl.item():.3f}")
        step += 1

print("✅ GRPO-style training loop finished.")

# --- (Optional) Save adapters and merged model ---
OUT_DIR = "smollm2_grpo_reasoning_loop"
import os
os.makedirs(OUT_DIR, exist_ok=True)
ADAPTER_DIR = os.path.join(OUT_DIR, "lora_adapter")
MERGED_DIR  = os.path.join(OUT_DIR, "merged")
os.makedirs(ADAPTER_DIR, exist_ok=True)
os.makedirs(MERGED_DIR, exist_ok=True)

policy.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

merged_model = policy.merge_and_unload()
try:
    from unsloth import unsloth_save_model
    try:
        unsloth_save_model(merged_model, MERGED_DIR)
    except TypeError:
        unsloth_save_model(model=merged_model, save_directory=MERGED_DIR)
except Exception:
    merged_model.save_pretrained(MERGED_DIR, safe_serialization=True)
tokenizer.save_pretrained(MERGED_DIR)
print("💾 Saved LoRA + merged model to:", OUT_DIR)


The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


step    0 | loss 0.0000 | reward_mean 0.000 | acc 0.00 | kl 0.000
step   10 | loss -0.0942 | reward_mean 0.000 | acc 0.00 | kl -4.719
step   20 | loss -0.8398 | reward_mean 0.000 | acc 0.00 | kl -42.000
step   30 | loss -3.5625 | reward_mean 0.000 | acc 0.00 | kl -178.000
step   40 | loss -24.7500 | reward_mean 0.000 | acc 0.00 | kl -1240.000
step   50 | loss -54.0000 | reward_mean 0.000 | acc 0.00 | kl -2704.000
step   60 | loss -53.5000 | reward_mean 0.000 | acc 0.00 | kl -2672.000
step   70 | loss -102.5000 | reward_mean 0.000 | acc 0.00 | kl -5120.000
step   80 | loss -93.0000 | reward_mean 0.000 | acc 0.00 | kl -4640.000
step   90 | loss -100.5000 | reward_mean 0.000 | acc 0.00 | kl -5024.000
step  100 | loss -118.0000 | reward_mean 0.000 | acc 0.00 | kl -5888.000
step  110 | loss -102.5000 | reward_mean 0.000 | acc 0.00 | kl -5120.000
step  120 | loss -211.0000 | reward_mean 0.000 | acc 0.00 | kl -10560.000
step  130 | loss -223.0000 | reward_mean 0.000 | acc 0.00 | kl -11136.000

In [9]:
# === Robust save: LoRA adapter + merged model (handles unsloth_save_model signature) ===
import os, inspect

OUT_DIR = "smollm2_grpo_reasoning"
ADAPTER_DIR = os.path.join(OUT_DIR, "lora_adapter")
MERGED_DIR  = os.path.join(OUT_DIR, "merged")
os.makedirs(ADAPTER_DIR, exist_ok=True)
os.makedirs(MERGED_DIR, exist_ok=True)

# 1) Save LoRA adapter-only
policy.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print("✅ Saved LoRA adapter →", ADAPTER_DIR)

# 2) Merge LoRA to standalone model
merged = policy.merge_and_unload()

# 3) Try Unsloth helper (both possible signatures), else fallback to HF save_pretrained
saved_via = None
try:
    from unsloth import unsloth_save_model
    sig = str(inspect.signature(unsloth_save_model))
    try:
        # Try positional (model, save_directory)
        unsloth_save_model(merged, MERGED_DIR)
        saved_via = "unsloth(positional)"
    except TypeError:
        # Try keyword version (model=..., save_directory=...)
        unsloth_save_model(model=merged, save_directory=MERGED_DIR)
        saved_via = "unsloth(kwargs)"
except Exception as e:
    pass

if not saved_via:
    # Fallback: standard Transformers save
    merged.save_pretrained(MERGED_DIR, safe_serialization=True)
    saved_via = "transformers.save_pretrained"

# 4) Save tokenizer to merged dir too
tokenizer.save_pretrained(MERGED_DIR)

print(f"✅ Saved merged model → {MERGED_DIR} via {saved_via}")


✅ Saved LoRA adapter → smollm2_grpo_reasoning/lora_adapter
✅ Saved merged model → smollm2_grpo_reasoning/merged via transformers.save_pretrained


In [10]:
from unsloth import FastLanguageModel

infer_model, infer_tok = FastLanguageModel.from_pretrained(
    model_name=MERGED_DIR, max_seq_length=MAX_SEQ_LEN, dtype=dtype, load_in_4bit=False
)
FastLanguageModel.for_inference(infer_model)

def ask(prompt):
    msgs = [{"role":"user","content": prompt}]
    text = infer_tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = infer_tok([text], return_tensors="pt").to(infer_model.device)
    out = infer_model.generate(**inputs, max_new_tokens=256, do_sample=True, temperature=0.7)
    print(infer_tok.decode(out[0], skip_special_tokens=True))

ask("If a train travels 60 km in 1.5 hours, what is its average speed?")


==((====))==  Unsloth 2025.11.2: Fast Llama patching. Transformers: 4.57.1.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.557 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu126. CUDA: 8.0. CUDA Toolkit: 12.6. Triton: 3.4.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.32.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
smollm2_grpo_reasoning/merged does not have a padding token! Will use pad_token = <|endoftext|>.
system
You are a helpful AI assistant named SmolLM, trained by Hugging Face
user
If a train travels 60 km in 1.5 hours, what is its average speed?
assistant
In simple terms, if we consider the train's average speed as an average of the total distance covered and the time of travel, we can solve this problem as follows:

1. Find the total distance covered by the train: 60 km in 1.5 hours (60 * 1.5 = 90 m)

2. Find the average speed